# Portfolio Risk Management - Exploration

Initial data exploration and factor analysis before building the full pipeline.

**Tickers used:** AAPL, MSFT, JPM, JNJ, XOM, GOOGL, AMZN, UNH, PG, V  
**Period:** 2021-2024

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup done')

## 1. Data Loading

In [ ]:
from src.data_loader import get_price_data, get_fundamentals, get_fred_data, get_fama_french_factors

tickers = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'XOM', 'GOOGL', 'AMZN', 'UNH', 'PG', 'V']
start = '2021-01-01'
end = '2024-12-31'

prices = get_price_data(tickers, start, end)
fundamentals = get_fundamentals(tickers)
fred_data = get_fred_data(start=start, end=end)

print(f'Price data: {prices.shape}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Tickers loaded: {list(prices.columns)}')

In [ ]:
# check for missing values in price data
missing = prices.isnull().sum()
print('Missing values per ticker:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

prices.head()

In [ ]:
# normalized price chart (base = 100)
normalized = (prices / prices.iloc[0]) * 100

fig = px.line(normalized, title='Normalized Prices (Base = 100)',
              labels={'value': 'Price (Base 100)', 'Date': ''})
fig.update_layout(height=500, legend_title_text='Ticker')
fig.show()

In [ ]:
# daily returns distribution
returns = prices.pct_change().dropna()

print('Daily returns summary:')
returns.describe().round(4)

In [ ]:
# histogram of daily returns for each ticker
fig = make_subplots(rows=2, cols=5, subplot_titles=tickers)

for i, ticker in enumerate(tickers):
    row = i // 5 + 1
    col = i % 5 + 1
    fig.add_trace(
        go.Histogram(x=returns[ticker], nbinsx=50, name=ticker, showlegend=False),
        row=row, col=col
    )

fig.update_layout(height=400, title='Daily Returns Distribution')
fig.show()

In [ ]:
# return correlation matrix
corr = returns.corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Return Correlation Matrix')
fig.update_layout(height=500, width=600)
fig.show()

In [ ]:
# annualized return and volatility per ticker
ann_ret = returns.mean() * 252
ann_vol = returns.std() * np.sqrt(252)

risk_return = pd.DataFrame({'Ann Return': ann_ret, 'Ann Volatility': ann_vol})
risk_return['Sharpe'] = (risk_return['Ann Return'] - 0.05) / risk_return['Ann Volatility']

print('Risk-Return Profile:')
risk_return.sort_values('Sharpe', ascending=False).round(4)

In [ ]:
# risk-return scatter
fig = px.scatter(risk_return, x='Ann Volatility', y='Ann Return', text=risk_return.index,
                 title='Risk-Return Scatter', size_max=10)
fig.update_traces(textposition='top center', marker=dict(size=10))
fig.update_layout(height=450, xaxis_title='Annualized Volatility', yaxis_title='Annualized Return')
fig.show()

## 2. Fundamentals

In [ ]:
# look at the fundamental data we pulled
print(f'Fundamentals shape: {fundamentals.shape}')
print(f'Columns: {list(fundamentals.columns)}')
fundamentals[['pb_ratio', 'pe_ratio', 'fcf_yield', 'roe', 'profit_margin', 'debt_to_equity', 'sector']]

In [ ]:
# valuation comparison
val_cols = ['pb_ratio', 'pe_ratio', 'fcf_yield']
fund_plot = fundamentals[val_cols].copy()
fund_plot = fund_plot.clip(upper=fund_plot.quantile(0.95))  # clip outliers for plotting

fig = make_subplots(rows=1, cols=3, subplot_titles=['P/B Ratio', 'P/E Ratio', 'FCF Yield'])

for i, col in enumerate(val_cols):
    fig.add_trace(
        go.Bar(x=fund_plot.index, y=fund_plot[col], name=col, showlegend=False),
        row=1, col=i+1
    )

fig.update_layout(height=350, title='Valuation Metrics')
fig.show()

## 3. Factor Scores

In [ ]:
from src.factor_model import (
    compute_value_score, compute_momentum_score, compute_quality_score,
    compute_composite_score, rank_stocks, run_factor_pipeline
)

# compute individual factor scores
value_scores = compute_value_score(fundamentals)
momentum_scores = compute_momentum_score(prices)
quality_scores = compute_quality_score(fundamentals)

scores_df = pd.DataFrame({
    'Value': value_scores,
    'Momentum': momentum_scores,
    'Quality': quality_scores
}).dropna()

print('Factor scores (z-scored):')
scores_df.round(3)

In [ ]:
# factor score bar chart
fig = go.Figure()
for col in scores_df.columns:
    fig.add_trace(go.Bar(x=scores_df.index, y=scores_df[col], name=col))

fig.update_layout(barmode='group', title='Factor Scores by Ticker',
                  height=450, yaxis_title='Z-Score')
fig.show()

In [ ]:
# factor correlation - checking if factors capture different info
factor_corr = scores_df.corr()
print('Factor Correlation:')
print(factor_corr.round(3))
print()

max_corr = factor_corr.where(np.triu(np.ones(factor_corr.shape), k=1).astype(bool)).stack().abs().max()
print(f'Max pairwise correlation: {max_corr:.3f}')
print(f'Factors are {"independent enough" if max_corr < 0.5 else "somewhat correlated"}')

In [ ]:
# composite score with equal weights
composite = compute_composite_score(value_scores, momentum_scores, quality_scores)
composite_df = composite.sort_values(ascending=False)

fig = px.bar(x=composite_df.index, y=composite_df.values,
             title='Composite Factor Score (Equal Weights)',
             labels={'x': 'Ticker', 'y': 'Composite Z-Score'},
             color=composite_df.values, color_continuous_scale='RdYlGn')
fig.update_layout(height=400)
fig.show()

## 4. Macro Regime Detection

In [ ]:
from src.macro_regime import get_current_regime, get_factor_weights, get_yield_curve_multiplier, describe_regime, run_macro_overlay

macro = run_macro_overlay(fred_data)

print(f'Detected regime: {macro["regime"]}')
print(f'Description: {macro["description"]}')
print(f'Yield curve multiplier: {macro["yield_curve_multiplier"]:.3f}')
print(f'Factor weights: {macro["factor_weights"]}')

In [ ]:
# fed funds rate over time
if 'FEDFUNDS' in fred_data.columns:
    fig = px.line(fred_data['FEDFUNDS'].dropna(), title='Fed Funds Rate',
                  labels={'value': 'Rate (%)', 'Date': ''})
    fig.update_layout(height=350)
    fig.show()

In [ ]:
# yield curve (10Y - 2Y spread)
if 'T10Y2Y' in fred_data.columns:
    spread = fred_data['T10Y2Y'].dropna()
    fig = px.area(spread, title='10Y-2Y Treasury Spread',
                  labels={'value': 'Spread (%)', 'Date': ''})
    fig.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Inversion')
    fig.update_layout(height=350)
    fig.show()

In [ ]:
# composite score with regime-adjusted weights
regime_weights = macro['factor_weights']
composite_adj = compute_composite_score(value_scores, momentum_scores, quality_scores, weights=regime_weights)

compare = pd.DataFrame({
    'Equal Weights': composite,
    'Regime Adjusted': composite_adj
}).dropna()

print('Ranking changes after regime adjustment:')
compare['Equal Rank'] = compare['Equal Weights'].rank(ascending=False).astype(int)
compare['Regime Rank'] = compare['Regime Adjusted'].rank(ascending=False).astype(int)
compare['Rank Change'] = compare['Equal Rank'] - compare['Regime Rank']
compare.sort_values('Regime Rank')

## 5. Portfolio Optimization

In [ ]:
from src.portfolio_optimizer import compute_expected_returns, compute_covariance_matrix, optimize_portfolio, get_efficient_frontier

exp_ret = compute_expected_returns(prices, factor_scores=composite_adj)
cov_mat = compute_covariance_matrix(prices)

print('Expected returns (annualized):')
print(exp_ret.sort_values(ascending=False).round(4))
print()
print('Covariance matrix shape:', cov_mat.shape)

In [ ]:
# optimize - max sharpe
result = optimize_portfolio(exp_ret, cov_mat, yield_multiplier=macro['yield_curve_multiplier'])

weights = result['weights']
print(f'Optimization: {result["objective"]}')
print(f'Expected return: {result["expected_return"]:.2%}')
print(f'Expected volatility: {result["expected_volatility"]:.2%}')
print(f'Sharpe ratio: {result["sharpe_ratio"]:.3f}')
print()
print('Weights:')
print(weights.sort_values(ascending=False).round(4))

In [ ]:
# portfolio weights pie chart
w = weights[weights > 0.001].sort_values(ascending=False)
fig = px.pie(values=w.values, names=w.index, title='Optimal Portfolio Weights')
fig.update_layout(height=400)
fig.show()

In [ ]:
# efficient frontier
frontier = get_efficient_frontier(exp_ret, cov_mat)

fig = go.Figure()
fig.add_trace(go.Scatter(x=frontier['volatility'], y=frontier['return'],
                         mode='lines', name='Efficient Frontier'))

# mark the optimal portfolio
fig.add_trace(go.Scatter(x=[result['expected_volatility']], y=[result['expected_return']],
                         mode='markers', name='Optimal', marker=dict(size=12, color='red')))

# mark individual stocks
stock_vols = returns.std() * np.sqrt(252)
stock_rets = returns.mean() * 252
common = [t for t in tickers if t in stock_vols.index]
fig.add_trace(go.Scatter(x=stock_vols[common], y=stock_rets[common],
                         mode='markers+text', name='Individual Stocks',
                         text=common, textposition='top center',
                         marker=dict(size=8, color='gray')))

fig.update_layout(title='Efficient Frontier', height=500,
                  xaxis_title='Annualized Volatility', yaxis_title='Annualized Return')
fig.show()

## 6. Backtest

In [ ]:
from src.backtester import run_backtest

bt = run_backtest(tickers, start_date=start, end_date=end, rebalance_freq='M')

meta = bt['metadata']
print(f'Backtest period: {meta["start_date"]} to {meta["end_date"]}')
print(f'Total return: {meta["total_return"]:.2%}')
print(f'Ann return: {meta["ann_return"]:.2%}')
print(f'Ann volatility: {meta["ann_volatility"]:.2%}')
print(f'Sharpe ratio: {meta["sharpe_ratio"]:.3f}')
print(f'Max drawdown: {meta["max_drawdown"]:.2%}')
print(f'Rebalances: {meta["n_rebalances"]}')
print(f'Avg turnover: {meta["avg_turnover"]:.2%}')

In [ ]:
# NAV vs benchmark
nav = bt['nav']
bench_ret = bt['benchmark_returns']
bench_nav = (1 + bench_ret).cumprod() * 100_000

fig = go.Figure()
fig.add_trace(go.Scatter(x=nav.index, y=nav.values, name='Portfolio'))
fig.add_trace(go.Scatter(x=bench_nav.index, y=bench_nav.values, name='SPY Benchmark'))
fig.update_layout(title='Portfolio NAV vs SPY', height=450,
                  yaxis_title='Value ($)', xaxis_title='')
fig.show()

In [ ]:
# drawdown chart
from src.risk_metrics import compute_drawdown_series

dd = compute_drawdown_series(bt['daily_returns'])

fig = px.area(dd, title='Portfolio Drawdown',
              labels={'value': 'Drawdown', 'Date': ''})
fig.update_layout(height=350)
fig.show()

In [ ]:
# portfolio weights over time
wh = bt['weights_history']
if wh:
    dates = [d for d, w in wh]
    weight_records = [w for d, w in wh]
    wt_df = pd.DataFrame(weight_records, index=dates).fillna(0)
    
    fig = px.area(wt_df, title='Portfolio Weights Over Time',
                  labels={'value': 'Weight', 'variable': 'Ticker'})
    fig.update_layout(height=450)
    fig.show()

## 7. Risk Metrics

In [ ]:
from src.risk_metrics import compute_all_metrics, compute_rolling_metrics, compute_fama_french_alpha, get_metrics_summary

metrics = compute_all_metrics(bt['daily_returns'], bt['benchmark_returns'])

summary = get_metrics_summary(metrics)
summary

In [ ]:
# rolling sharpe and volatility
rolling = compute_rolling_metrics(bt['daily_returns'], bt['benchmark_returns'], window=126)  # 6-month rolling

fig = make_subplots(rows=2, cols=1, subplot_titles=['Rolling Sharpe (6M)', 'Rolling Volatility (6M)'],
                    shared_xaxes=True, vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=rolling.index, y=rolling['rolling_sharpe'], name='Sharpe'), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_trace(go.Scatter(x=rolling.index, y=rolling['rolling_volatility'], name='Volatility'), row=2, col=1)

fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# fama-french 3-factor regression
ff_factors = get_fama_french_factors(start=start, end=end)
ff_result = compute_fama_french_alpha(bt['daily_returns'], ff_factors)

print('Fama-French 3-Factor Alpha:')
print(f'  Monthly alpha: {ff_result["alpha_monthly"]:.4f}')
print(f'  Annualized alpha: {ff_result["alpha_annual"]:.2%}')
print(f'  Market beta: {ff_result["beta_mkt"]:.3f}')
print(f'  SMB beta: {ff_result["beta_smb"]:.3f}')
print(f'  HML beta: {ff_result["beta_hml"]:.3f}')
print(f'  R-squared: {ff_result["r_squared"]:.3f}')
print(f'  Alpha t-stat: {ff_result["t_stat_alpha"]:.3f}')
print(f'  Alpha p-value: {ff_result["p_value_alpha"]:.4f}')
print(f'  Significant at 5%: {ff_result["significant_5pct"]}')

## 8. Monte Carlo Simulation

In [ ]:
from src.monte_carlo import run_monte_carlo, compute_var_cvar, get_simulation_summary

# run 10k simulations, 1 year forward
sim_paths = run_monte_carlo(weights, prices, n_simulations=10_000, horizon_days=252)

sim_summary = get_simulation_summary(sim_paths)
print(f'Initial value: ${sim_summary["initial_value"]:,.0f}')
print(f'Median final: ${sim_summary["median_final"]:,.0f}')
print(f'Mean final: ${sim_summary["mean_final"]:,.0f}')
print(f'5th percentile: ${sim_summary["percentile_5"]:,.0f}')
print(f'95th percentile: ${sim_summary["percentile_95"]:,.0f}')
print(f'Prob of profit: {sim_summary["prob_profit"]:.1%}')
print(f'Prob of >20% loss: {sim_summary["prob_loss_20pct"]:.1%}')

In [ ]:
# VaR and CVaR from simulation
var_cvar = compute_var_cvar(sim_paths, confidence=0.95)
print(f'95% VaR: ${var_cvar["var_dollar"]:,.0f} ({var_cvar["var_pct"]:.2%})')
print(f'95% CVaR: ${var_cvar["cvar_dollar"]:,.0f} ({var_cvar["cvar_pct"]:.2%})')

In [ ]:
# fan chart - plot 200 random paths + percentile bands
fig = go.Figure()

# sample 200 paths
np.random.seed(42)
sample_idx = np.random.choice(sim_paths.shape[0], 200, replace=False)
for idx in sample_idx:
    fig.add_trace(go.Scatter(y=sim_paths[idx], mode='lines',
                             line=dict(width=0.3, color='lightblue'),
                             showlegend=False, hoverinfo='skip'))

# percentile bands
p5 = np.percentile(sim_paths, 5, axis=0)
p50 = np.percentile(sim_paths, 50, axis=0)
p95 = np.percentile(sim_paths, 95, axis=0)

fig.add_trace(go.Scatter(y=p50, mode='lines', name='Median', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(y=p5, mode='lines', name='5th pctl', line=dict(color='red', dash='dash')))
fig.add_trace(go.Scatter(y=p95, mode='lines', name='95th pctl', line=dict(color='green', dash='dash')))

fig.update_layout(title='Monte Carlo Simulation (10,000 paths, 1 year)',
                  height=500, xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)')
fig.show()

In [ ]:
# distribution of final portfolio values
final_values = sim_paths[:, -1]

fig = go.Figure()
fig.add_trace(go.Histogram(x=final_values, nbinsx=100, name='Final Value'))
fig.add_vline(x=100_000, line_dash='dash', line_color='red', annotation_text='Initial')
fig.add_vline(x=np.median(final_values), line_dash='dash', line_color='blue', annotation_text='Median')

fig.update_layout(title='Distribution of Final Portfolio Values',
                  height=400, xaxis_title='Portfolio Value ($)', yaxis_title='Count')
fig.show()

## 9. Stress Testing

In [ ]:
from src.stress_test import run_all_scenarios, get_scenario_summary

stress_results = run_all_scenarios(weights)
stress_summary = get_scenario_summary(stress_results)

print('Stress Test Results:')
stress_summary

In [ ]:
# stress test bar chart
fig = px.bar(stress_summary, x='portfolio_return', y='scenario',
             orientation='h', title='Portfolio Loss Under Historical Crises',
             labels={'portfolio_return': 'Portfolio Return', 'scenario': ''},
             color='portfolio_return', color_continuous_scale='RdYlGn')
fig.update_layout(height=350)
fig.show()

## 10. Statistical Validation

In [ ]:
from src.hypothesis_testing import (
    test_ic_significance, test_strategy_significance, 
    test_factor_independence, generate_validation_report, get_ic_summary_table
)

# information coefficient tests
ic_results = test_ic_significance(prices, fundamentals)
ic_table = get_ic_summary_table(ic_results)

print('IC Significance Test:')
ic_table

In [ ]:
# bootstrap test - is strategy performance due to skill or luck?
bootstrap = test_strategy_significance(bt['daily_returns'], bt['benchmark_returns'])

print('Bootstrap Strategy Test:')
print(f'  Actual Sharpe: {bootstrap["actual_sharpe"]:.3f}')
print(f'  Bootstrap mean Sharpe: {bootstrap["bootstrap_mean_sharpe"]:.3f}')
print(f'  Bootstrap 5th-95th: [{bootstrap["bootstrap_5th_pctl"]:.3f}, {bootstrap["bootstrap_95th_pctl"]:.3f}]')
print(f'  p-value (Sharpe): {bootstrap["p_value_sharpe"]:.4f}')
print(f'  Significant at 5%: {bootstrap["significant_5pct"]}')

if 'p_value_excess' in bootstrap:
    print(f'  Excess return p-value: {bootstrap["p_value_excess"]:.4f}')
    print(f'  Excess return significant: {bootstrap["excess_significant_5pct"]}')

In [ ]:
# factor independence
independence = test_factor_independence(value_scores, momentum_scores, quality_scores)

print('Factor Independence:')
print(f'  Value-Momentum corr: {independence["value_momentum"]["correlation"]:.3f} (p={independence["value_momentum"]["p_value"]:.4f})')
print(f'  Value-Quality corr: {independence["value_quality"]["correlation"]:.3f} (p={independence["value_quality"]["p_value"]:.4f})')
print(f'  Momentum-Quality corr: {independence["momentum_quality"]["correlation"]:.3f} (p={independence["momentum_quality"]["p_value"]:.4f})')
print(f'  Max |correlation|: {independence["max_abs_correlation"]:.3f}')
print(f'  Independent: {independence["factors_independent"]}')

In [ ]:
# overall validation report
report = generate_validation_report(ic_results, bootstrap, independence)

print(f'Overall verdict: {report["overall_verdict"]}')
print(f'  IC verdict: {report["ic_verdict"]}')
print(f'  Bootstrap verdict: {report["bootstrap_verdict"]}')
print(f'  Independence verdict: {report["independence_verdict"]}')

## Summary

Key findings from this analysis:

- Three factor signals (Value, Momentum, Quality) show low cross-correlation, confirming they capture different information
- Macro regime overlay adjusts factor weights based on Fed policy — currently in a rate environment that tilts toward quality/defensive names
- Mean-variance optimizer produces a diversified portfolio with max 15% single-stock exposure
- Monte Carlo simulations give a distribution of outcomes for risk budgeting
- Stress tests show expected losses under historical crisis scenarios
- Statistical tests validate whether factor signals and strategy performance are meaningful